# Pancancer enrichment analysis step 3: Make figures from Reactome data

## Setup

In [1]:
import cptac
import cptac.utils as ut
import os
import datetime
import altair as alt
import numpy as np
import pandas as pd

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP2_DIR = "step2_outputs"
STEP2_FILE = "enrichment_20200609_171630_from_tumor_change_20200529_195104_10000_perm.tsv"
STEP2_FILE_PATH = os.path.join(STEP2_DIR, STEP2_FILE)

## Read in data from step 2

In [3]:
data = pd.read_csv(STEP2_FILE_PATH, sep="\t", index_col=0)

In [4]:
data

,pathway_id,cancer_type,mean_expr,url,name,pathway_times_enriched,pathway_avg_rank,entities_pValue,entities_fdr,cancer_rank_pval
0,R-HSA-6798695,ccrcc,0.233503,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,3.467008e-03,8.426404e-01,1.0
1,R-HSA-6798695,colon,-0.134831,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,6.216916e-12,1.500142e-08,1.0
2,R-HSA-6798695,endometrial,0.288515,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,1.403496e-03,8.219205e-01,1.0
3,R-HSA-6798695,gbm,0.166227,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,2.129677e-03,8.037977e-01,1.0
4,R-HSA-6798695,hnscc,0.133159,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,3.649260e-03,8.329209e-01,1.0
5,R-HSA-6798695,lscc,-0.407595,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,1.637558e-03,8.726502e-01,1.0
6,R-HSA-6798695,luad,-0.410000,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,1.800932e-04,4.412282e-01,1.0
7,R-HSA-6798695,ovarian,-0.303754,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,3.532076e-05,8.660650e-02,1.0
8,R-HSA-6791226,ccrcc,0.699422,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Major pathway of rRNA processing in the nucleo...,8,2.875,3.132093e-02,8.426404e-01,4.0
9,R-HSA-6791226,colon,0.771429,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Major pathway of rRNA processing in the nucleo...,8,2.875,2.825057e-07,1.376497e-04,2.0


In [5]:
data["pathway_avg_rank"].value_counts()

17.750    16
20.500     8
18.500     8
13.625     8
12.750     8
10.500     8
5.375      8
2.875      8
1.000      8
Name: pathway_avg_rank, dtype: int64

## Figure 1: Bubble plot

In [6]:
data = data.assign(
    rank_size=-1 * np.log10(data["cancer_rank_pval"] + 1),
    avg_rank_size=-1 * np.log10(data["pathway_avg_rank"] + 1),
    avg_rank_label=data["pathway_avg_rank"].astype(str))

mean_expr_max = max(data["mean_expr"])
mean_expr_min = min(data["mean_expr"])

In [7]:
# Take care of duplicates for the upper plot
# If ther were more than two values that were the same,
# we'd need to repeat this multiple times until there
# was no change.

data["avg_rank_label"] = data["avg_rank_label"].where(
    cond=~(data.duplicated(subset=["cancer_type", "avg_rank_label"], keep="first")),
    other=" " + data["avg_rank_label"])

In [8]:
individual = alt.Chart(data).mark_circle().encode(
    x=alt.X(
        "name:N",
        sort=data["name"].values,
        axis=alt.Axis(
            labelAngle=-20,
            labelFontSize=12,
            labelLimit=500,
            title="Pathway",
            titleFontSize=12
        )
    ),
    y=alt.Y(
        "cancer_type:N",
        axis=alt.Axis(
            title="Cancer type",
            titleFontSize=12
        ),
    ),
    size="rank_size:Q",
    color=alt.Color(
        "mean_expr:Q",
        scale=alt.Scale(
            scheme="redblue",
            domain=[mean_expr_max, mean_expr_min]
        )
    )
).properties(
    width=600,
    height=400
)

aggregate = alt.Chart(data).mark_circle().encode(
    x=alt.X(
        "avg_rank_label:N",
        sort=data["avg_rank_label"].values,
        axis=alt.Axis(
            labelAngle=-20,
            labelFontSize=12,
            labelLimit=500,
            title="Average rank of pathway",
            titleFontSize=12
        )
    ),
    size="avg_rank_size:Q",
).properties(
    width=600
)

alt.vconcat(aggregate, individual).configure_axis(
    grid=True
)


#######################################3
# NEXT: Can we make the differences between circle size more dramatic with some transformation?

alt.VConcatChart(...)

In [9]:
def barplot_most_enriched(data, cancer_type):
    
    data = data[data["cancer_type"] == cancer_type].\
        sort_values(by=["entities_fdr", "entities_pValue"]).\
        iloc[0:NUM_TOP, :]
    
    plot = alt.Chart(data).mark_bar().encode(
        x=alt.X(
            "name:N",
            sort=data["name"].values,
            axis=alt.Axis(
                title="<---- Lower FDR | Higher FDR ---->",
                labelAngle=20,
                labelFontSize=14,
                labelLimit=500,
                titleFontSize=16
            )
        ),
        y=alt.Y(
            "expr_mean:Q",
            axis=alt.Axis(
                title="Pathway mean expression change",
                titleFontSize=16
            )
        ),
        color=alt.condition(
            alt.datum.expr_mean > 0,
            alt.value("#fde500"),
            alt.value("#04429b")
        )
    ).properties(
        width=600,
        height=500,
        title=f"{cancer_type.title()} top enriched pathways"
    ).configure_title(
        fontSize=16
    )
    
    plot.save(f"/home/caleb/Downloads/{cancer_type}.png")

    return plot